In [7]:
import numpy as np
from PIL import Image
import os
from collections import Counter
import random

class DecisionTree:
    """Решающее дерево для случайного леса"""
    def __init__(self, max_depth=10, min_samples_split=2, n_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        # Если не задано количество фичей, используем все
        self.n_features = X.shape[1] if self.n_features is None else min(self.n_features, X.shape[1])
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Критерии остановки
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # Выбираем случайные фичи
        feat_idxs = random.sample(range(n_features), self.n_features)

        # Ищем лучшее разбиение
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # Создаем дочерние узлы
        left_idxs = np.argwhere(X[:, best_feat] <= best_thresh).flatten()
        right_idxs = np.argwhere(X[:, best_feat] > best_thresh).flatten()
        
        left = self._grow_tree(X[left_idxs], y[left_idxs], depth+1)
        right = self._grow_tree(X[right_idxs], y[right_idxs], depth+1)
        
        return Node(best_feat, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        best_gini = float('inf')
        split_idx, split_thresh = None, None
        
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for threshold in thresholds:
                gini = self._gini(y, X_column, threshold)
                
                if gini < best_gini:
                    best_gini = gini
                    split_idx = feat_idx
                    split_thresh = threshold

        return split_idx, split_thresh

    def _gini(self, y, X_column, split_thresh):
        left_mask = X_column <= split_thresh
        right_mask = X_column > split_thresh
        
        n_left = np.sum(left_mask)
        n_right = np.sum(right_mask)
        n_total = n_left + n_right
        
        if n_left == 0 or n_right == 0:
            return float('inf')
        
        gini_left = 1 - sum((np.sum(y[left_mask] == c) / n_left) ** 2 for c in np.unique(y))
        gini_right = 1 - sum((np.sum(y[right_mask] == c) / n_right) ** 2 for c in np.unique(y))
        
        return (n_left * gini_left + n_right * gini_right) / n_total

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


class Node:
    """Узел дерева"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        
    def is_leaf(self):
        return self.value is not None


class RandomForest:
    """Случайный лес для классификации изображений"""
    def __init__(self, n_trees=100, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                n_features=self.n_features
            )
            
            X_sample, y_sample = self._bootstrap_sample(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, size=n_samples, replace=True)
        return X[idxs], y[idxs]

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        
        return np.array([self._most_common_label(pred) for pred in tree_preds])

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]


def extract_features(image_path):
    """Извлекает признаки из изображения"""
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img) / 255.0
    
    brightness = np.mean(img_array)
    green_ratio = np.mean(img_array[:, :, 1]) 
    contrast = np.std(img_array)
    
    return np.array([brightness, green_ratio, contrast])


def load_dataset(dataset_path):
    """Загружает датасет изображений"""
    X, y = [], []
    for class_name in ['forest', 'desert']:
        class_dir = os.path.join(dataset_path, class_name)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            features = extract_features(img_path)
            X.append(features)
            y.append(0 if class_name == 'forest' else 1)  # 0 - лес, 1 - пустыня
    
    return np.array(X), np.array(y)

In [8]:
# Путь к датасету
DATASET_PATH = "origins/task_2"

# Загрузка данных
X, y = load_dataset(DATASET_PATH)

# Объединяем и перемешиваем
data = list(zip(X, y))
random.shuffle(data)  
X_shuffled, y_shuffled = zip(*data) 

# Разделение на обучающую и тестовую выборки
split_idx = int(0.8 * len(X_shuffled))
X_train, y_train = X[:split_idx], y[:split_idx]
X_test, y_test = X[split_idx:], y[split_idx:]

# Создание и обучение случайного леса
rf = RandomForest(n_trees=10, max_depth=10, n_features=3)
rf.fit(X_train, y_train)

# Предсказание на тестовых данных
y_pred = rf.predict(X_test)

# Оценка точности
accuracy = np.mean(y_pred == y_test)
print(f"Точность случайного леса: {accuracy * 100:.2f}%")

Точность случайного леса: 83.85%
